<a href="https://colab.research.google.com/github/Maziger/Laksegate-master-thesis/blob/main/POC/tabicl_experiment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TabICL Experiment 2 — Forecast Combination for Electricity Price

**Goal:** Use `TabICLRegressor` as a *stacking* / *forecast combination* model.  
The inputs are the 10 DNN and LEAR ensemble price forecasts; the target is the true German day-ahead electricity price.

This differs from Experiment 1 (univariate `TabICLForecaster`) by treating the problem as supervised tabular regression: TabICL learns how to best combine/correct pre-existing ensemble forecasts, rather than forecasting from raw time series.

**Dataset:** `Forecasts_DE_DNN_LEAR_ensembles.csv` — 17 472 hourly observations (2016-01-04 → 2017-12-31)

**Features (X)**
- DNN 1, DNN 2, DNN 3, DNN 4, DNN Ensemble
- LEAR 56, LEAR 84, LEAR 1092, LEAR 1456, LEAR Ensemble
- Hour-of-day, day-of-week (calendar effects)
- Lagged actual price at −24 h and −168 h (yesterday / last week, same hour)

**Target (y):** Real price (€/MWh)

code: https://github.com/soda-inria/tabicl  
paper: https://arxiv.org/abs/2602.11139

In [ ]:
import os
from google.colab import userdata

user = "Maziger"
repo = "Laksegate-master-thesis"

# remove local directory if it already exists
if os.path.isdir(repo):
    !rm -rf {repo}

!git clone https://github.com/{user}/{repo}.git
%cd Laksegate-master-thesis/

!pip install -q tabicl

## Data loading & feature engineering

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "POC/Forecasts_DE_DNN_LEAR_ensembles.csv",
    index_col=0,
    parse_dates=True,
)
df.index.name = "timestamp"

ENSEMBLE_COLS = [
    "DNN 1", "DNN 2", "DNN 3", "DNN 4", "DNN Ensemble",
    "LEAR 56", "LEAR 84", "LEAR 1092", "LEAR 1456", "LEAR Ensemble",
]
TARGET = "Real price"

# --- Calendar features ---
df["hour"]    = df.index.hour
df["weekday"] = df.index.weekday   # 0 = Monday, 6 = Sunday

# --- Lag features of the actual price ---
# These represent what a forecaster would know at prediction time
df["price_lag_24"]  = df[TARGET].shift(24)   # yesterday, same hour
df["price_lag_168"] = df[TARGET].shift(168)  # last week, same hour

# Drop the first 168 rows that have NaN lags
df = df.dropna()

FEATURE_COLS = ENSEMBLE_COLS + ["hour", "weekday", "price_lag_24", "price_lag_168"]

print(f"Total samples after dropping NaN lags: {len(df)}")
print(f"Date range: {df.index[0]} → {df.index[-1]}")
df[FEATURE_COLS + [TARGET]].head(3)

In [ ]:
# Chronological 80/20 train-test split (no shuffling — time series)
n = len(df)
n_train = int(n * 0.8)

train_df = df.iloc[:n_train]
test_df  = df.iloc[n_train:]

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET]
X_test  = test_df[FEATURE_COLS]
y_test  = test_df[TARGET]

print(f"Train: {len(X_train)} rows  ({train_df.index[0].date()} → {train_df.index[-1].date()})")
print(f"Test:  {len(X_test)} rows  ({test_df.index[0].date()} → {test_df.index[-1].date()})")

## Baseline models

Before TabICL we compute two naive baselines:
- **DNN Ensemble** — the pre-computed DNN ensemble forecast used directly as prediction
- **LEAR Ensemble** — the pre-computed LEAR ensemble forecast used directly as prediction

These represent the current state of the art the dataset was built around.

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

def mape(y_true, y_pred):
    """Mean Absolute Percentage Error, ignoring near-zero actuals."""
    mask = np.abs(y_true) > 1.0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    m    = mape(y_true.values, np.asarray(y_pred))
    print(f"{name:<22}  MAE={mae:6.3f}  RMSE={rmse:6.3f}  MAPE={m:5.2f}%")
    return {"MAE": mae, "RMSE": rmse, "MAPE": m}

results = {}
results["DNN Ensemble"]  = evaluate("DNN Ensemble",  y_test, X_test["DNN Ensemble"])
results["LEAR Ensemble"] = evaluate("LEAR Ensemble", y_test, X_test["LEAR Ensemble"])

## TabICLRegressor — Forecast Combination

`TabICLRegressor` receives all 10 ensemble forecasts plus calendar and lag features and learns to combine them into a single, improved price prediction via in-context learning — no hyperparameter tuning required.

In [ ]:
import time
from tabicl import TabICLRegressor

regressor = TabICLRegressor(
    n_estimators=8,
    random_state=42,
)

t0 = time.time()
regressor.fit(X_train, y_train)
print(f"Fit completed in {time.time() - t0:.1f}s")

In [ ]:
t0 = time.time()
y_pred = regressor.predict(X_test, output_type="mean")
print(f"Prediction completed in {time.time() - t0:.1f}s")

results["TabICL"] = evaluate("TabICL", y_test, y_pred)

## Results summary

In [ ]:
results_df = pd.DataFrame(results).T
results_df.index.name = "Model"
results_df = results_df.sort_values("MAE")
results_df.style.format("{:.3f}").background_gradient(cmap="RdYlGn_r")

## Visualisation — predictions vs actual price

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

# Plot a representative 2-week window from the test set
WINDOW = 24 * 14   # 14 days
plot_idx   = test_df.index[:WINDOW]
y_true_win = y_test.values[:WINDOW]
y_pred_win = y_pred[:WINDOW]
dnn_win    = X_test["DNN Ensemble"].values[:WINDOW]
lear_win   = X_test["LEAR Ensemble"].values[:WINDOW]

plt.ioff()
fig, ax = plt.subplots(figsize=(15, 5))

ax.plot(plot_idx, y_true_win, label="Actual",        color="black",  linewidth=1.5)
ax.plot(plot_idx, y_pred_win, label="TabICL",        color="tab:blue",   linewidth=1.5)
ax.plot(plot_idx, dnn_win,  label="DNN Ensemble",  color="tab:orange", linewidth=1.0, linestyle="--", alpha=0.7)
ax.plot(plot_idx, lear_win, label="LEAR Ensemble", color="tab:green",  linewidth=1.0, linestyle="--", alpha=0.7)

ax.set_title("Electricity Price Forecast Combination — first 14 days of test set")
ax.set_xlabel("Date")
ax.set_ylabel("Price (€/MWh)")
ax.legend()
ax.grid(True, alpha=0.4)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
fig.autofmt_xdate()

display(fig)
plt.close(fig)

## Prediction intervals

TabICL natively supports quantile outputs. The 10th–90th percentile band gives an 80% prediction interval — useful for risk-aware trading decisions.

In [ ]:
quantiles = regressor.predict(X_test, output_type="quantiles", alphas=[0.10, 0.90])
q_lo, q_hi = quantiles[:, 0], quantiles[:, 1]

# Empirical coverage
coverage = np.mean((y_test.values >= q_lo) & (y_test.values <= q_hi))
print(f"Empirical 80% prediction interval coverage: {coverage:.1%}")

plt.ioff()
fig, ax = plt.subplots(figsize=(15, 5))

ax.fill_between(
    plot_idx,
    q_lo[:WINDOW],
    q_hi[:WINDOW],
    alpha=0.25,
    color="tab:blue",
    label="80% PI (TabICL)",
)
ax.plot(plot_idx, y_true_win, label="Actual",  color="black",    linewidth=1.5)
ax.plot(plot_idx, y_pred_win, label="TabICL",  color="tab:blue", linewidth=1.5)

ax.set_title("TabICL Forecast Combination with 80% Prediction Interval")
ax.set_xlabel("Date")
ax.set_ylabel("Price (€/MWh)")
ax.legend()
ax.grid(True, alpha=0.4)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
fig.autofmt_xdate()

display(fig)
plt.close(fig)